# Notebook 6 — Mechanism check: input heterogeneity vs explicit income cue

This notebook extends the preregistered Claude mechanism experiment without recollecting any Claude data.

## Core design

The original mechanism experiment used **Claude Sonnet 5** on a fixed sample of 2,000 WVS respondents:

1. **Observed-income condition** — reuses the original Claude predictions from Notebook 2.
2. **No-income condition** — Q288 removed from the profile.
3. **Counterfactual income sweep** — the same 2,000 respondents are each assigned Q288 = 1, ..., 10 while all other profile characteristics are held fixed.

The existing Claude mechanism predictions are treated as **frozen**. This revised notebook will **not make new Claude API calls by default**.

## Cross-model extension

To test whether the no-income result generalizes beyond Claude, the notebook adds a **no-income arm only** for:

- Gemini 3.8 Flash
- DeepSeek V4.1 Flash

Each model receives the same fixed 2,000-person mechanism sample with Q288 removed. This adds only 2,000 calls per model.

## Safety

`ALLOW_NEW_CLAUDE_CALLS = False` by default.

If the existing Claude mechanism output file is missing, the notebook will stop rather than silently recollect Claude data. To recollect Claude intentionally, you would have to change that flag manually.


In [ ]:
# Run once in Deepnote if needed.
%pip install -q "anthropic>=0.70" "google-genai>=1.0" "openai>=1.100" pandas numpy matplotlib tqdm


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import random
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# -------------------------
# Frozen input / sample
# -------------------------

FULL_PROFILE_FILE = Path("output/full_wvs/profiles_for_prediction.csv")
CLAUDE_FULL_PREDICTIONS = Path("output/full_wvs/predictions/anthropic_sonnet5_FULL.csv")

OUTPUT_DIR = Path("output/full_wvs/mechanism_claude")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_FULL_N = 93901
EXPECTED_PROFILE_VERSION = "wvs7_v6_profiles_v2_2026-09-22"

SAMPLE_N = 2000
SAMPLE_SEED = 20261015
STRATIFY_ON_ORIGINAL_INCOME = True

PILOT_MODE = False
PILOT_N = 20

MAX_WORKERS = 10
BATCH_SIZE = 500
RETRIES = 6
CHECKPOINT_EVERY = 100

N_BOOT = 1000
BOOT_SEED = 20261015

MECHANISM_VERSION = "income_mechanism_multimodel_v2_2026-10-17"

# -------------------------
# Model configuration
# -------------------------

MODEL_CONFIG = {
    "claude": {
        "provider": "anthropic",
        "model_id": "claude-sonnet-5",
        "api_key_env": "ANTHROPIC_API_KEY",
    },
    "gemini": {
        "provider": "google_gemini",
        "model_id": "gemini-3.8-flash",
        "api_key_env": "GEMINI_API_KEY",
    },
    "deepseek": {
        "provider": "deepseek",
        "model_id": "deepseek-flash",
        "api_key_env": "DEEPSEEK_API_KEY",
    },
}

# SAFETY SWITCH:
# Existing Claude mechanism data should be reused, not recollected.
ALLOW_NEW_CLAUDE_CALLS = False

# Only these new no-income replications are enabled by default.
RUN_NEW_MODELS = ["gemini", "deepseek"]

# Safe reruns: do not make one-off connectivity calls unless explicitly requested.
RUN_CONNECTIVITY_TESTS = False

SYSTEM_PROMPT = (
    "You will be given a World Values Survey respondent profile. "
    "Answer the life-satisfaction question as that person would most likely answer it. "
    "Return ONLY one integer from 1 to 10. Do not explain your answer."
)

QUESTION = (
    "All things considered, how satisfied are you with your life as a whole these days? "
    "Using a scale on which 1 means you are 'completely dissatisfied' and 10 means you are "
    "'completely satisfied', where would you put your satisfaction with your life as a whole?"
)

Q288_WORDING = (
    "Household income group in your country, counting wages, salaries, pensions and other "
    "household income (1-Lowest income group, 10-Highest income group)"
)

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def build_prompt(profile):
    return (
        f"You are a person with the following profile:\n\n{profile}"
        f"\n\nQuestion:\n{QUESTION}"
    )


## 1. Reconstruct the exact same fixed 2,000-person mechanism sample

The sample logic is unchanged from the preregistered Claude experiment: 200 respondents are selected from each original Q288 category using the same fixed seed.

The saved sample file is reused if present. If it exists, its row IDs are checked against the deterministic reconstruction.


In [ ]:
if not FULL_PROFILE_FILE.exists():
    raise FileNotFoundError(f"Missing {FULL_PROFILE_FILE}. Run Notebook 1 in full mode first.")

df = pd.read_csv(FULL_PROFILE_FILE, low_memory=False)

required_cols = [
    "WVS_ROW_ID", "B_COUNTRY_ALPHA", "Q49", "Q288",
    "user_description", "PROFILE_VERSION"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

if len(df) != EXPECTED_FULL_N:
    raise ValueError(f"Expected {EXPECTED_FULL_N:,} rows; found {len(df):,}.")

if not df["WVS_ROW_ID"].is_unique:
    raise AssertionError("WVS_ROW_ID is not unique.")

full_income_counts = df["Q288"].astype(int).value_counts().sort_index()
full_income_shares = full_income_counts / len(df)

if SAMPLE_N is None:
    sample_reconstructed = df.copy()
elif STRATIFY_ON_ORIGINAL_INCOME:
    if SAMPLE_N % 10 != 0:
        raise ValueError("SAMPLE_N must be divisible by 10.")
    n_per_group = SAMPLE_N // 10
    pieces = []
    for q in range(1, 11):
        g = df.loc[df["Q288"].eq(q)]
        pieces.append(g.sample(n=n_per_group, random_state=SAMPLE_SEED + q))
    sample_reconstructed = pd.concat(pieces, ignore_index=True)
else:
    sample_reconstructed = df.sample(n=SAMPLE_N, random_state=SAMPLE_SEED).copy()

sample_reconstructed = (
    sample_reconstructed.sort_values("WVS_ROW_ID").reset_index(drop=True)
)

sample_income_counts = (
    sample_reconstructed["Q288"].astype(int).value_counts().sort_index()
)
sample_income_shares = sample_income_counts / len(sample_reconstructed)

weight_map = {
    q: float(full_income_shares.loc[q] / sample_income_shares.loc[q])
    for q in range(1, 11)
}
sample_reconstructed["analysis_weight"] = (
    sample_reconstructed["Q288"].astype(int).map(weight_map).astype(float)
)

if PILOT_MODE:
    pilot_per_group = max(1, PILOT_N // 10)
    sample_reconstructed = (
        sample_reconstructed
        .groupby(sample_reconstructed["Q288"].astype(int), group_keys=False)
        .head(pilot_per_group)
        .head(PILOT_N)
        .copy()
        .reset_index(drop=True)
    )

SAMPLE_FILE = OUTPUT_DIR / (
    "mechanism_sample_PILOT.csv"
    if PILOT_MODE else
    f"mechanism_sample_N{len(sample_reconstructed)}.csv"
)

if SAMPLE_FILE.exists():
    sample_saved = pd.read_csv(SAMPLE_FILE, low_memory=False)
    saved_ids = sample_saved["WVS_ROW_ID"].astype(int).tolist()
    reconstructed_ids = sample_reconstructed["WVS_ROW_ID"].astype(int).tolist()
    if saved_ids != reconstructed_ids:
        raise AssertionError(
            "Saved mechanism sample does not match deterministic reconstruction. "
            "Stop before collecting any new model data."
        )
    sample = sample_saved.copy()
    print("Reusing existing frozen mechanism sample:", SAMPLE_FILE)
else:
    sample = sample_reconstructed.copy()
    sample.to_csv(SAMPLE_FILE, index=False)
    print("Saved frozen mechanism sample:", SAMPLE_FILE)

print("Mechanism N:", len(sample))
print("Countries represented:", sample["B_COUNTRY_ALPHA"].nunique())
print("Sample SHA256:", sha256_file(SAMPLE_FILE))


## 2. Profile manipulation

The no-income arm removes only the Q288 clause from `user_description`. The counterfactual sweep helper is retained for auditing the frozen Claude experiment, but the new Gemini and DeepSeek runs use **no-income only**.


In [ ]:
Q288_CLAUSE_RE = re.compile(
    r"(?:,\s*)?"
    + re.escape("the person answered the question ")
    + r"'"
    + re.escape(Q288_WORDING)
    + r"' as '\d+(?:\.0+)? - \d+(?:\.0+)?'"
)

def count_q288_clause(profile):
    return len(Q288_CLAUSE_RE.findall(str(profile)))

def remove_q288(profile):
    profile = str(profile)
    hits = count_q288_clause(profile)
    if hits != 1:
        raise ValueError(f"Expected exactly 1 Q288 clause; found {hits}.")
    edited = Q288_CLAUSE_RE.sub("", profile, count=1)
    edited = edited.replace("where ,", "where ")
    edited = re.sub(r"\s{2,}", " ", edited)
    edited = edited.replace(" ,", ",")
    return edited.strip()

def set_q288(profile, assigned_income):
    assigned_income = int(assigned_income)
    if assigned_income not in range(1, 11):
        raise ValueError("assigned_income must be 1..10.")
    profile = str(profile)
    hits = count_q288_clause(profile)
    if hits != 1:
        raise ValueError(f"Expected exactly 1 Q288 clause; found {hits}.")
    replacement = (
        "the person answered the question "
        f"'{Q288_WORDING}' as "
        f"'{assigned_income} - {assigned_income}'"
    )
    m = Q288_CLAUSE_RE.search(profile)
    prefix = ", " if m.group(0).lstrip().startswith(",") else ""
    return (profile[:m.start()] + prefix + replacement + profile[m.end():]).strip()

counts = sample["user_description"].map(count_q288_clause)
if not counts.eq(1).all():
    raise AssertionError("At least one sampled profile does not contain exactly one Q288 clause.")

no_income_jobs = sample[
    ["WVS_ROW_ID", "B_COUNTRY_ALPHA", "Q288", "analysis_weight", "user_description"]
].copy()
no_income_jobs["condition"] = "no_income"
no_income_jobs["profile_for_api"] = no_income_jobs["user_description"].map(remove_q288)
no_income_jobs["job_id"] = no_income_jobs["WVS_ROW_ID"].map(
    lambda x: f"no_income__{int(x)}"
)

print("No-income jobs per new model:", len(no_income_jobs))


## 3. Provider API functions

These mirror the provider-specific settings used in Notebook 2.

- Gemini: same model ID and system instruction; no artificial output-token ceiling.
- DeepSeek: same model ID; thinking disabled; 16-token cap.
- Claude: code is retained only for compatibility. New Claude calls are blocked unless `ALLOW_NEW_CLAUDE_CALLS=True`.


In [ ]:
def require_env(name):
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing environment variable: {name}")
    return value

def parse_score(text):
    if text is None:
        raise ValueError("Model returned no visible text.")
    text = str(text).strip()
    if re.fullmatch(r"10|[1-9]", text):
        return int(text)
    nums = re.findall(r"(?<!\d)(10|[1-9])(?!\d)", text)
    if len(nums) == 1:
        return int(nums[0])
    raise ValueError(f"Could not parse a unique 1-10 score from: {text!r}")

_CLIENTS = {}

def reset_clients():
    global _CLIENTS
    _CLIENTS = {}

def get_client(model_key):
    cfg = MODEL_CONFIG[model_key]
    provider = cfg["provider"]

    if model_key in _CLIENTS:
        return _CLIENTS[model_key]

    if provider == "anthropic":
        import anthropic
        client = anthropic.Anthropic(
            api_key=require_env(cfg["api_key_env"])
        )

    elif provider == "google_gemini":
        from google import genai
        client = genai.Client(
            api_key=require_env(cfg["api_key_env"])
        )

    elif provider == "deepseek":
        from openai import OpenAI
        client = OpenAI(
            api_key=require_env(cfg["api_key_env"]),
            base_url="https://api.deepseek.com",
        )

    else:
        raise ValueError(provider)

    _CLIENTS[model_key] = client
    return client

def call_model(profile, model_key):
    cfg = MODEL_CONFIG[model_key]
    provider = cfg["provider"]
    model = cfg["model_id"]

    if model_key == "claude" and not ALLOW_NEW_CLAUDE_CALLS:
        raise RuntimeError(
            "New Claude calls are disabled. Existing Claude mechanism data must be reused."
        )

    client = get_client(model_key)
    user_text = build_prompt(profile)

    if provider == "anthropic":
        r = client.messages.create(
            model=model,
            max_tokens=16,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_text}],
        )
        raw = "".join(
            b.text for b in r.content
            if getattr(b, "type", "") == "text"
        )

    elif provider == "google_gemini":
        from google.genai import types
        r = client.models.generate_content(
            model=model,
            contents=user_text,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
            ),
        )
        raw = r.text

    elif provider == "deepseek":
        r = client.chat.completions.create(
            model=model,
            messages=[
                {"role":"system", "content":SYSTEM_PROMPT},
                {"role":"user", "content":user_text},
            ],
            max_tokens=16,
            extra_body={"thinking":{"type":"disabled"}},
        )
        raw = r.choices[0].message.content

    else:
        raise ValueError(provider)

    return parse_score(raw), str(raw).strip()


## 4. Load the frozen Claude mechanism results

This is the key protection against accidental recollection.

The notebook expects the original Claude mechanism output generated by the prior Notebook 6 run. If it is missing, execution stops.

No Claude API call occurs in this section.


In [ ]:
CLAUDE_MECHANISM_FILE = OUTPUT_DIR / (
    "claude_mechanism_predictions_PILOT.csv"
    if PILOT_MODE else
    f"claude_mechanism_predictions_N{len(sample)}.csv"
)

if not CLAUDE_MECHANISM_FILE.exists():
    raise FileNotFoundError(
        f"Frozen Claude mechanism file not found: {CLAUDE_MECHANISM_FILE}\n"
        "This revised notebook will NOT recollect Claude automatically."
    )

claude_mechanism = pd.read_csv(CLAUDE_MECHANISM_FILE)

required = {
    "job_id", "WVS_ROW_ID", "country", "original_income",
    "condition", "prediction", "status"
}
missing = required - set(claude_mechanism.columns)
if missing:
    raise ValueError(f"Frozen Claude mechanism file is missing columns: {sorted(missing)}")

claude_mechanism = claude_mechanism.loc[
    claude_mechanism["status"].eq("ok")
].copy()

n_expected_claude = len(sample) * 11
if len(claude_mechanism) != n_expected_claude:
    raise AssertionError(
        f"Frozen Claude file has {len(claude_mechanism):,} successful rows; "
        f"expected {n_expected_claude:,}."
    )

claude_ids = set(claude_mechanism["WVS_ROW_ID"].astype(int))
sample_ids = set(sample["WVS_ROW_ID"].astype(int))
if claude_ids != sample_ids:
    raise AssertionError("Frozen Claude mechanism file does not match the fixed mechanism sample.")

print("Frozen Claude mechanism data loaded successfully.")
print("Rows:", len(claude_mechanism))
print("No new Claude calls will be made.")


## 5. Optional connectivity tests for the **new models only**

These make one call each for Gemini and DeepSeek.

Claude is intentionally excluded.


In [ ]:
if RUN_CONNECTIVITY_TESTS:
    for model_key in RUN_NEW_MODELS:
        score, raw = call_model(
            no_income_jobs.iloc[0]["profile_for_api"],
            model_key
        )
        print(model_key, "parsed score:", score, "| raw:", repr(raw))
else:
    print("Connectivity tests skipped (RUN_CONNECTIVITY_TESTS=False).")


## 6. Run / resume Gemini and DeepSeek no-income arms

Each model gets its own checkpoint file.

If a run is interrupted, rerunning the cell skips successful `job_id`s.

The file naming is model-specific, so these runs cannot overwrite the frozen Claude mechanism file.


In [ ]:
def predict_job(row, model_key):
    last_error = None

    for attempt in range(1, RETRIES + 1):
        try:
            score, raw = call_model(row["profile_for_api"], model_key)
            return {
                "job_id": row["job_id"],
                "WVS_ROW_ID": int(row["WVS_ROW_ID"]),
                "country": row["B_COUNTRY_ALPHA"],
                "original_income": int(row["Q288"]),
                "condition": "no_income",
                "analysis_weight": float(row["analysis_weight"]),
                "prediction": int(score),
                "raw_response": raw,
                "status": "ok",
                "error": None,
                "attempts": attempt,
            }
        except Exception as e:
            last_error = repr(e)
            if attempt < RETRIES:
                time.sleep(min(60, (2 ** (attempt - 1)) + random.random()))

    return {
        "job_id": row["job_id"],
        "WVS_ROW_ID": int(row["WVS_ROW_ID"]),
        "country": row["B_COUNTRY_ALPHA"],
        "original_income": int(row["Q288"]),
        "condition": "no_income",
        "analysis_weight": float(row["analysis_weight"]),
        "prediction": np.nan,
        "raw_response": None,
        "status": "error",
        "error": last_error,
        "attempts": RETRIES,
    }

def model_no_income_file(model_key):
    suffix = "PILOT" if PILOT_MODE else f"N{len(sample)}"
    return OUTPUT_DIR / f"{model_key}_no_income_{suffix}.csv"

def run_no_income_model(model_key):
    if model_key == "claude":
        raise RuntimeError(
            "Claude is frozen and must not be passed to run_no_income_model()."
        )

    out_file = model_no_income_file(model_key)

    records = []
    done = set()

    if out_file.exists():
        existing = pd.read_csv(out_file)
        records = existing.to_dict("records")
        done = set(
            existing.loc[
                existing["status"].eq("ok"),
                "job_id"
            ].astype(str)
        )
        print(
            f"{model_key}: resuming with "
            f"{len(done):,}/{len(no_income_jobs):,} successful jobs already present."
        )

    todo = no_income_jobs.loc[
        ~no_income_jobs["job_id"].isin(done)
    ].copy()

    print(f"{model_key}: jobs remaining = {len(todo):,}")

    completed_since_checkpoint = 0

    def checkpoint():
        nonlocal records
        out = pd.DataFrame(records)
        if out.empty:
            return
        out = (
            out.drop_duplicates("job_id", keep="last")
            .sort_values("WVS_ROW_ID")
            .reset_index(drop=True)
        )
        out["model_key"] = model_key
        out["model_id"] = MODEL_CONFIG[model_key]["model_id"]
        out["mechanism_version"] = MECHANISM_VERSION
        out["profile_version"] = EXPECTED_PROFILE_VERSION
        out["sample_sha256"] = sha256_file(SAMPLE_FILE)
        out.to_csv(out_file, index=False)

    for start in range(0, len(todo), BATCH_SIZE):
        batch = todo.iloc[start:start + BATCH_SIZE]

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = {
                ex.submit(predict_job, row, model_key): row["job_id"]
                for _, row in batch.iterrows()
            }

            for fut in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"{model_key}: {start+1}-{start+len(batch)}"
            ):
                records.append(fut.result())
                completed_since_checkpoint += 1

                if completed_since_checkpoint >= CHECKPOINT_EVERY:
                    checkpoint()
                    completed_since_checkpoint = 0

        checkpoint()

    checkpoint()

    result = pd.read_csv(out_file)
    result = result.loc[
        result["job_id"].isin(set(no_income_jobs["job_id"]))
    ].copy()

    n_ok = int(result["status"].eq("ok").sum())
    n_err = int(result["status"].eq("error").sum())

    print(f"{model_key}: successful {n_ok:,}/{len(no_income_jobs):,}")
    print(f"{model_key}: errors {n_err:,}")

    if n_err:
        display(
            result.loc[
                result["status"].eq("error"),
                ["job_id", "error"]
            ].head(20)
        )
        raise RuntimeError(
            f"{model_key}: failed rows remain. "
            "Rerun this cell to retry only failed jobs."
        )

    if len(result) != len(no_income_jobs):
        raise AssertionError(
            f"{model_key}: got {len(result):,} rows; "
            f"expected {len(no_income_jobs):,}."
        )

    return result

new_model_results = {}

for model_key in RUN_NEW_MODELS:
    new_model_results[model_key] = run_no_income_model(model_key)


## 7. Reuse observed-income predictions for Claude, Gemini and DeepSeek

The original full-sample predictions from Notebook 2 are reused for all three models where available.

No API calls are made here.


In [ ]:
ORIGINAL_PREDICTION_FILES = {
    "claude": Path("output/full_wvs/predictions/anthropic_sonnet5_FULL.csv"),
    "gemini": Path("output/full_wvs/predictions/google_gemini38_flash_FULL.csv"),
    "deepseek": Path("output/full_wvs/predictions/deepseek_v41_flash_FULL.csv"),
}

observed_by_model = {}

for model_key, pred_file in ORIGINAL_PREDICTION_FILES.items():
    if not pred_file.exists():
        print(model_key, ": original prediction file not found:", pred_file)
        continue

    p = pd.read_csv(pred_file)
    if not {"WVS_ROW_ID", "prediction", "status"}.issubset(p.columns):
        raise ValueError(f"{pred_file} is missing required columns.")

    p = p.loc[
        p["status"].eq("ok"),
        ["WVS_ROW_ID", "prediction"]
    ].copy()

    p["WVS_ROW_ID"] = p["WVS_ROW_ID"].astype(int)

    obs = sample[
        ["WVS_ROW_ID", "B_COUNTRY_ALPHA", "Q288", "analysis_weight"]
    ].merge(
        p,
        on="WVS_ROW_ID",
        how="left",
        validate="one_to_one"
    )

    if obs["prediction"].isna().any():
        raise AssertionError(
            f"{model_key}: missing observed-income prediction(s) "
            "for the fixed mechanism sample."
        )

    obs = obs.rename(
        columns={
            "B_COUNTRY_ALPHA":"country",
            "Q288":"original_income"
        }
    )
    obs["condition"] = "observed_income"
    observed_by_model[model_key] = obs

    print(model_key, ": observed-income predictions loaded =", len(obs))


## 8. Analysis helpers

All profiles are normalized by each source's own weighted-average SD across the ten income groups, using the same full-sample human Q288 shares as weights. This matches the main paper's scale-adjusted structural-fidelity metric.


In [ ]:
def weighted_mean(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    return float(np.average(x[mask], weights=w[mask]))

def weighted_sd(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x, w = x[mask], w[mask]
    mu = np.average(x, weights=w)
    return float(np.sqrt(np.average((x - mu)**2, weights=w)))

def summarize_by_group(data, group_col):
    rows = []
    for g, d in data.groupby(group_col, sort=True):
        rows.append({
            group_col: int(g),
            "N": len(d),
            "mean": weighted_mean(d["prediction"], d["analysis_weight"]),
            "sd": weighted_sd(d["prediction"], d["analysis_weight"]),
        })
    out = pd.DataFrame(rows).sort_values(group_col).reset_index(drop=True)
    return out

def normalize_profile(summary, group_col):
    q = summary[group_col].astype(int).to_numpy()
    sds = summary["sd"].to_numpy(float)
    shares = np.array([full_income_shares.loc[int(x)] for x in q], dtype=float)
    scale = np.average(sds, weights=shares)
    out = summary.copy()
    out["normalized_sd"] = out["sd"] / scale
    out["normalization_scale"] = scale
    return out

def linear_slope(summary, x_col, y_col):
    x = summary[x_col].to_numpy(float)
    y = summary[y_col].to_numpy(float)
    return float(np.polyfit(x, y, 1)[0])

def quadratic_coefficients(summary, x_col, y_col):
    x = summary[x_col].to_numpy(float)
    y = summary[y_col].to_numpy(float)
    # y = a*x^2 + b*x + c
    a, b, c = np.polyfit(x, y, 2)
    return float(a), float(b), float(c)


## Human benchmark profile

Construct the normalized human Q49 dispersion profile using the same full-sample human Q288 shares used throughout the paper.


In [ ]:

human_for_profile = df[
    ["Q49", "Q288"]
].copy()
human_for_profile["analysis_weight"] = 1.0
human_for_profile["prediction"] = human_for_profile["Q49"].astype(float)
human_for_profile["original_income"] = human_for_profile["Q288"].astype(int)

human_summary = normalize_profile(
    summarize_by_group(human_for_profile, "original_income"),
    "original_income"
)

display(human_summary)


## 9. Build Claude, Gemini and DeepSeek no-income summaries

Claude's no-income data come from the frozen mechanism output.

Gemini and DeepSeek use the newly collected no-income files.

The full Claude counterfactual sweep is retained for the original preregistered mechanism test.


In [ ]:
# Claude frozen arms
claude_no_income = claude_mechanism.loc[
    claude_mechanism["condition"].eq("no_income")
].copy()

claude_sweep = claude_mechanism.loc[
    claude_mechanism["condition"].eq("income_sweep")
].copy()

claude_no_income_summary = normalize_profile(
    summarize_by_group(claude_no_income, "original_income"),
    "original_income"
)

claude_sweep_summary = normalize_profile(
    summarize_by_group(claude_sweep, "assigned_income"),
    "assigned_income"
)

# New models
no_income_summary_by_model = {
    "claude": claude_no_income_summary
}

for model_key, result in new_model_results.items():
    s = normalize_profile(
        summarize_by_group(result, "original_income"),
        "original_income"
    )
    no_income_summary_by_model[model_key] = s

display(pd.concat(
    [
        s.assign(model=model_key)
        for model_key, s in no_income_summary_by_model.items()
    ],
    ignore_index=True
))


## 10. Plot normalized no-income profiles across models

This is the cross-model extension: does the income-related dispersion gradient remain when Q288 is hidden from each model?


In [ ]:
fig = plt.figure()
fig.set_size_inches(7, 4.5)
fig.set_dpi(150)
ax = fig.add_subplot(111)

for model_key, s in no_income_summary_by_model.items():
    ax.plot(
        s["original_income"],
        s["normalized_sd"],
        marker="o",
        label=model_key
    )

ax.axhline(1, linewidth=1)
ax.set_xlabel("Actual Q288 income position (not shown to model)")
ax.set_ylabel("Normalized SD of predicted life satisfaction")
ax.set_title("No-income ablation across models")
ax.set_xticks(range(1, 11))
ax.legend()
fig.tight_layout()
plt.show()


## 11. Claude counterfactual sweep: report both linear slope and curvature

The preregistered primary sweep test is the linear SD slope.

Because the full profile is visibly U-shaped, this notebook also reports a **post-preregistration quadratic coefficient** as a descriptive characterization of convexity. The linear result remains the preregistered primary estimand.


In [ ]:
claude_sweep_raw_sd_slope = linear_slope(
    claude_sweep_summary,
    "assigned_income",
    "sd"
)

claude_sweep_norm_sd_slope = linear_slope(
    claude_sweep_summary,
    "assigned_income",
    "normalized_sd"
)

quad_a_raw, quad_b_raw, quad_c_raw = quadratic_coefficients(
    claude_sweep_summary,
    "assigned_income",
    "sd"
)

quad_a_norm, quad_b_norm, quad_c_norm = quadratic_coefficients(
    claude_sweep_summary,
    "assigned_income",
    "normalized_sd"
)

print("Claude sweep raw SD slope:", claude_sweep_raw_sd_slope)
print("Claude sweep normalized SD slope:", claude_sweep_norm_sd_slope)
print("Claude sweep raw quadratic term:", quad_a_raw)
print("Claude sweep normalized quadratic term:", quad_a_norm)

fig = plt.figure()
fig.set_size_inches(7, 4.5)
fig.set_dpi(150)
ax = fig.add_subplot(111)

ax.plot(
    claude_sweep_summary["assigned_income"],
    claude_sweep_summary["normalized_sd"],
    marker="o"
)
ax.axhline(1, linewidth=1)
ax.set_xlabel("Counterfactually assigned Q288 income position")
ax.set_ylabel("Normalized SD of predicted life satisfaction")
ax.set_title("Claude counterfactual income sweep — normalized dispersion")
ax.set_xticks(range(1, 11))
fig.tight_layout()
plt.show()


## 12. Paired country-cluster bootstrap

This section adds the comparisons that matter most:

- Claude observed-income slope
- Claude no-income slope
- Claude sweep slope
- observed minus no-income slope
- sweep minus no-income slope

It also estimates no-income slopes for Gemini and DeepSeek.

Countries are resampled with replacement. The same resampled-country multiplicities are used across paired Claude arms.


In [ ]:

def summarize_with_boot_weight(data, group_col, weight_col="boot_weight"):
    rows = []
    for g, d in data.groupby(group_col, sort=True):
        rows.append({
            group_col: int(g),
            "sd": weighted_sd(d["prediction"], d[weight_col]),
        })
    return pd.DataFrame(rows).sort_values(group_col).reset_index(drop=True)

def normalize_summary_boot(summary, group_col):
    q = summary[group_col].astype(int).to_numpy()
    shares = np.array([full_income_shares.loc[int(x)] for x in q], dtype=float)
    scale = np.average(summary["sd"].to_numpy(float), weights=shares)
    out = summary.copy()
    out["normalized_sd"] = out["sd"] / scale
    return out

def boot_profile(data, multiplicity, group_col):
    d = data.loc[
        data["country"].astype(str).isin(multiplicity.index.astype(str))
    ].copy()
    d["boot_weight"] = (
        d["analysis_weight"]
        * d["country"].astype(str).map(multiplicity).astype(float)
    )
    return normalize_summary_boot(
        summarize_with_boot_weight(d, group_col),
        group_col
    )

# Exploratory substantive-equivalence margin.
# This was NOT preregistered and should not be presented as a confirmatory equivalence test.
EXPLORATORY_EQUIV_MARGIN = 0.02

# Assemble no-income data for all three models.
model_no_income_data = {"claude": claude_no_income.copy()}
for k, v in new_model_results.items():
    model_no_income_data[k] = v.copy()

# Require observed predictions for all three models if their files were found.
models_for_paired = [
    k for k in ["claude", "gemini", "deepseek"]
    if k in observed_by_model and k in model_no_income_data
]

countries = np.array(sorted(sample["B_COUNTRY_ALPHA"].astype(str).unique()))
rng = np.random.default_rng(BOOT_SEED)
x = np.arange(1, 11, dtype=float)

boot_rows = []

for b in tqdm(range(N_BOOT), desc="Country-cluster bootstrap"):
    drawn = rng.choice(countries, size=len(countries), replace=True)
    multiplicity = pd.Series(drawn).value_counts()

    r = {"rep": b}
    valid = True

    # Paired observed-vs-no-income analysis for each model.
    for model_key in models_for_paired:
        po = boot_profile(
            observed_by_model[model_key],
            multiplicity,
            "original_income"
        )
        pn = boot_profile(
            model_no_income_data[model_key],
            multiplicity,
            "original_income"
        )

        if len(po) != 10 or len(pn) != 10:
            valid = False
            break

        yo = po["normalized_sd"].to_numpy(float)
        yn = pn["normalized_sd"].to_numpy(float)

        so = float(np.polyfit(x, yo, 1)[0])
        sn = float(np.polyfit(x, yn, 1)[0])

        r[f"{model_key}_observed_slope"] = so
        r[f"{model_key}_no_income_slope"] = sn
        r[f"{model_key}_observed_minus_no_income"] = so - sn
        r[f"{model_key}_observed_no_income_rmsd"] = float(
            np.sqrt(np.mean((yo - yn) ** 2))
        )

    if not valid:
        continue

    # Claude sweep: linear slope + post-preregistration curvature diagnostics.
    ps = boot_profile(
        claude_sweep,
        multiplicity,
        "assigned_income"
    )
    if len(ps) != 10:
        continue

    ys = ps["normalized_sd"].to_numpy(float)
    sweep_slope = float(np.polyfit(x, ys, 1)[0])
    sweep_curv = float(np.polyfit(x, ys, 2)[0])
    sweep_end_ratio = float(((ys[0] + ys[-1]) / 2.0) / ys.min())

    r["claude_sweep_slope"] = sweep_slope
    r["claude_sweep_curvature"] = sweep_curv
    r["claude_sweep_end_to_min_ratio"] = sweep_end_ratio

    if "claude_no_income_slope" in r:
        r["claude_sweep_minus_no_income"] = (
            r["claude_sweep_slope"] - r["claude_no_income_slope"]
        )

    boot_rows.append(r)

boot = pd.DataFrame(boot_rows)

if len(boot) < 0.95 * N_BOOT:
    raise RuntimeError(
        f"Only {len(boot)}/{N_BOOT} bootstrap replications were valid."
    )

def percentile_ci(x, alpha=0.05):
    x = pd.Series(x).dropna().to_numpy(float)
    return (
        float(np.quantile(x, alpha/2)),
        float(np.quantile(x, 1-alpha/2))
    )

def two_sided_boot_p(x, null=0.0):
    x = pd.Series(x).dropna().to_numpy(float)
    return min(
        1.0,
        2 * min(np.mean(x <= null), np.mean(x >= null))
    )

summary_rows = []

estimands = []
for k in models_for_paired:
    estimands.extend([
        f"{k}_observed_slope",
        f"{k}_no_income_slope",
        f"{k}_observed_minus_no_income",
        f"{k}_observed_no_income_rmsd",
    ])

estimands.extend([
    "claude_sweep_slope",
    "claude_sweep_curvature",
    "claude_sweep_end_to_min_ratio",
    "claude_sweep_minus_no_income",
])

for col in estimands:
    if col not in boot.columns:
        continue

    vals = boot[col].dropna()
    lo, hi = percentile_ci(vals)

    row = {
        "estimand": col,
        "bootstrap_mean": float(vals.mean()),
        "ci_low": lo,
        "ci_high": hi,
        "valid_reps": int(vals.notna().sum()),
    }

    # p-values are meaningful for signed slope/contrast/curvature estimands,
    # not for inherently nonnegative RMSD or end-to-minimum ratio.
    if not col.endswith("_rmsd") and not col.endswith("_ratio"):
        row["bootstrap_p"] = two_sided_boot_p(vals)
    else:
        row["bootstrap_p"] = np.nan

    if col.endswith("observed_minus_no_income"):
        row["exploratory_margin"] = EXPLORATORY_EQUIV_MARGIN
        row["ci_inside_exploratory_margin"] = bool(
            lo > -EXPLORATORY_EQUIV_MARGIN
            and hi < EXPLORATORY_EQUIV_MARGIN
        )
    else:
        row["exploratory_margin"] = np.nan
        row["ci_inside_exploratory_margin"] = np.nan

    summary_rows.append(row)

bootstrap_summary = pd.DataFrame(summary_rows)
display(bootstrap_summary)

print("\nObserved-minus-no-income contrasts:")
for k in models_for_paired:
    col = f"{k}_observed_minus_no_income"
    vals = boot[col].dropna()
    lo, hi = percentile_ci(vals)
    status = (
        "inside exploratory ±0.02 margin"
        if (lo > -EXPLORATORY_EQUIV_MARGIN and hi < EXPLORATORY_EQUIV_MARGIN)
        else "NOT inside exploratory ±0.02 margin"
    )
    print(
        f"{k}: mean={vals.mean():.4f}, "
        f"95% CI [{lo:.4f}, {hi:.4f}] — {status}"
    )

BOOT_FILE = OUTPUT_DIR / (
    "multimodel_no_income_bootstrap_PILOT.csv"
    if PILOT_MODE else
    f"multimodel_no_income_bootstrap_N{len(sample)}.csv"
)
boot.to_csv(BOOT_FILE, index=False)
print("Saved:", BOOT_FILE)


## Cross-model observed vs no-income profiles

Each panel overlays:

- the human normalized Q49 dispersion profile;
- the model's original observed-income profile;
- the model's no-income profile.

These plots make the cross-model ablation comparison directly visible.


In [ ]:

profile_comparison_rows = []

for model_key in models_for_paired:
    obs_summary = normalize_profile(
        summarize_by_group(
            observed_by_model[model_key],
            "original_income"
        ),
        "original_income"
    )

    noinc_summary = no_income_summary_by_model[model_key].copy()

    # Save compact comparison table.
    tmp = pd.DataFrame({
        "income_group": np.arange(1, 11),
        "human_normalized_sd": human_summary["normalized_sd"].to_numpy(float),
        "observed_normalized_sd": obs_summary["normalized_sd"].to_numpy(float),
        "no_income_normalized_sd": noinc_summary["normalized_sd"].to_numpy(float),
    })
    tmp["model"] = model_key
    tmp["observed_minus_no_income"] = (
        tmp["observed_normalized_sd"] - tmp["no_income_normalized_sd"]
    )
    profile_comparison_rows.append(tmp)

    fig = plt.figure()
    fig.set_size_inches(7, 4.5)
    fig.set_dpi(150)
    ax = fig.add_subplot(111)

    ax.plot(
        human_summary["original_income"],
        human_summary["normalized_sd"],
        marker="o",
        label="Human"
    )
    ax.plot(
        obs_summary["original_income"],
        obs_summary["normalized_sd"],
        marker="o",
        label=f"{model_key}: observed income"
    )
    ax.plot(
        noinc_summary["original_income"],
        noinc_summary["normalized_sd"],
        marker="o",
        label=f"{model_key}: income removed"
    )

    ax.axhline(1, linewidth=1)
    ax.set_xlabel("Actual Q288 income position")
    ax.set_ylabel("Normalized SD of life satisfaction")
    ax.set_title(f"{model_key.title()}: observed vs no-income profile")
    ax.set_xticks(range(1, 11))
    ax.legend()
    fig.tight_layout()
    plt.show()

profile_comparison = pd.concat(
    profile_comparison_rows,
    ignore_index=True
)
display(profile_comparison)

PROFILE_COMPARISON_FILE = OUTPUT_DIR / (
    "observed_vs_no_income_profiles_PILOT.csv"
    if PILOT_MODE else
    f"observed_vs_no_income_profiles_N{len(sample)}.csv"
)
profile_comparison.to_csv(PROFILE_COMPARISON_FILE, index=False)
print("Saved:", PROFILE_COMPARISON_FILE)


## 13. Save cross-model summaries


In [ ]:
all_summary = []

for model_key, s in no_income_summary_by_model.items():
    tmp = s.copy()
    tmp["model"] = model_key
    tmp["condition"] = "no_income"
    all_summary.append(tmp)

tmp = claude_sweep_summary.copy()
tmp = tmp.rename(columns={"assigned_income":"original_income"})
tmp["model"] = "claude"
tmp["condition"] = "counterfactual_income_sweep"
all_summary.append(tmp)

all_summary = pd.concat(all_summary, ignore_index=True)

SUMMARY_FILE = OUTPUT_DIR / (
    "multimodel_mechanism_summary_PILOT.csv"
    if PILOT_MODE else
    f"multimodel_mechanism_summary_N{len(sample)}.csv"
)

BOOT_SUMMARY_FILE = OUTPUT_DIR / (
    "multimodel_bootstrap_summary_PILOT.csv"
    if PILOT_MODE else
    f"multimodel_bootstrap_summary_N{len(sample)}.csv"
)

all_summary.to_csv(SUMMARY_FILE, index=False)
bootstrap_summary.to_csv(BOOT_SUMMARY_FILE, index=False)

print("Saved:")
print(" -", SUMMARY_FILE)
print(" -", BOOT_SUMMARY_FILE)
print(" -", BOOT_FILE)


## Interpretation guardrails

1. **Claude is not recollected.** Its existing mechanism data are frozen and reused.
2. Gemini and DeepSeek use the already-collected or resumable **no-income arm only**; their observed-income predictions are reused from the original full-sample runs.
3. The separately preregistered Claude mechanism analysis included the linear sweep and no-income analyses. The quadratic sweep term and end-to-minimum ratio are **post-preregistration exploratory diagnostics** motivated by the visible U-shape.
4. The ±0.02 margin is an **exploratory substantive-equivalence criterion only**. It was not preregistered and should not be described as a confirmatory equivalence test.
5. For each model, the most relevant cross-model comparison is the paired difference between its observed-income and no-income normalized slopes, supplemented by the profile RMSD.
6. Gemini and DeepSeek do not replicate the full Claude counterfactual sweep; they provide cross-model evidence only for the income-removal ablation.
7. The strongest defensible claim is that the negative income-related dispersion gradient can persist even when the explicit Q288 income cue is hidden from the model.
8. The Claude sweep should not be described as showing 'no effect of income on dispersion': the linear slope is near zero, but the full profile is visibly nonlinear and should be reported.
9. Do not claim that the analysis identifies internal neural mechanisms or an additive causal decomposition of the remaining profile characteristics.
